# Preparación de los datos

## Filtrado de columnas

crear un nuevo DataFrame que contenga únicamente las columnas que decidimos conservar en el diccionario de datos de la Etapa 2, descartando las que marcamos para eliminar (identificadores, geográficas, constantes, y las de data leakage como Churn Score y Churn Reason).

In [1]:
import pandas as pd

df=pd.read_excel('../data/raw/Telco_customer_churn.xlsx', sheet_name='Telco_Churn')

In [2]:
columnas_a_conservar = [
    'Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months',
    'Phone Service', 'Multiple Lines', 'Internet Service', 'Online Security',
    'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV',
    'Streaming Movies', 'Contract', 'Paperless Billing', 'Payment Method',
    'Monthly Charges', 'Total Charges', 'Churn Value'
]

df_limpio = df[columnas_a_conservar].copy()
df_limpio.shape

(7043, 20)

### Resultado: filtrado de columnas

Se creó `df_limpio`, un DataFrame con 20 columnas (19 predictoras + `Churn Value` como
target), descartando las 13 columnas identificadas en la Etapa 2 como no aptas para el
modelo (identificadores, geográficas, constantes, y las de data leakage `Churn Score` y
`Churn Reason`). Dimensiones confirmadas: (7043, 20).

## Corrección de Total Charges

Se convierte `Total Charges` de texto a numérico, imputando con 0 los 11 casos vacíos
identificados en la Etapa 2 (todos correspondientes a clientes con `Tenure Months = 0`,
recién dados de alta sin historial de facturación). Un modelo de Machine Learning no puede
operar matemáticamente sobre una columna tipada como texto, y la imputación con 0 está
justificada por la evidencia ya confirmada, no es una decisión arbitraria.

In [3]:
df_limpio['Total Charges'] = pd.to_numeric(df_limpio['Total Charges'], errors='coerce')
df_limpio['Total Charges'] = df_limpio['Total Charges'].fillna(0)

In [4]:
df_limpio['Total Charges'].dtype

dtype('float64')

In [5]:
df_limpio['Total Charges'].isnull().sum()

np.int64(0)

### Resultado: Total Charges corregida

`Total Charges` fue convertida exitosamente a `float64`. Los 11 casos vacíos fueron
imputados con 0, confirmado por `.isnull().sum() = 0`. La columna queda lista para
ser usada como variable numérica en el modelo.

## Encoding de variables binarias

Voy a convertir las columnas que tienen exactamente 2 valores posibles (Yes/No, Male/Female) a formato numérico 0/1.

Por qué como ya vimos con `Churn Value`/`Churn Label`, los modelos de scikit-learn necesitan números, no texto. Para columnas binarias, la transformación es la más simple posible — no hace falta crear columnas nuevas (como en one-hot), simplemente mapeamos cada categoría a 0 o 1.

Las columnas binarias que identificamos en la Etapa 2 son: `Gender`, `Senior Citizen`, `Partner`, `Dependents`, `Phone Service`, `Paperless Billing`

In [6]:
df_limpio['Gender'] = df_limpio['Gender'].map({'Male': 1, 'Female': 0})
df_limpio['Senior Citizen'] = df_limpio['Senior Citizen'].map({'Yes': 1, 'No': 0})
df_limpio['Partner'] = df_limpio['Partner'].map({'Yes': 1, 'No': 0})
df_limpio['Dependents'] = df_limpio['Dependents'].map({'Yes': 1, 'No': 0})
df_limpio['Phone Service'] = df_limpio['Phone Service'].map({'Yes': 1, 'No': 0})
df_limpio['Paperless Billing'] = df_limpio['Paperless Billing'].map({'Yes': 1, 'No': 0})

In [7]:
df_limpio[['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Paperless Billing']].isnull().sum()

Gender               0
Senior Citizen       0
Partner              0
Dependents           0
Phone Service        0
Paperless Billing    0
dtype: int64

### Resultado: encoding de variables binarias

Las 6 columnas binarias (Gender, Senior Citizen, Partner, Dependents, Phone Service,
Paperless Billing) fueron convertidas exitosamente a formato 0/1 mediante `.map()`,
sin generar valores nulos — confirmando que todos los valores de texto matchearon
correctamente con los diccionarios de mapeo.

## Decisión sobre las columnas "No internet service"
Debo decidir qué hacer con las 6 columnas que tienen ese tercer valor "No internet service" (`Online Security`, `Online Backup`, `Device Protection`, `Tech Support`, `Streaming TV`, `Streaming Movies`) — antes de aplicarles el encoding definitivo.

Por qué: ya confirmamos en la Etapa 2 que esas 1.526 filas con "No internet service" son 100% redundantes con `Internet Service = No` — es la misma información repetida en 6 columnas distintas. Si dejamos esa categoría separada en cada una y hacemos one-hot, estaríamos generando columnas casi idénticas entre sí (todas marcando "1" exactamente en las mismas 1.526 filas), lo cual no le aporta información nueva al modelo — solo ruido y redundancia.

In [8]:
columnas_servicios = ['Online Security','Online Backup','Device Protection','Tech Support','Streaming TV','Streaming Movies']

for col in columnas_servicios:
    df_limpio[col] = df_limpio[col].map({'Yes': 1, 'No': 0, 'No internet service': 0})

In [9]:
for col in columnas_servicios:
    print(f"Valores únicos en la columna '{col}':")
    print(df_limpio[col].value_counts())
    print()  # Línea en blanco para separar la salida de cada columna

Valores únicos en la columna 'Online Security':
Online Security
0    5024
1    2019
Name: count, dtype: int64

Valores únicos en la columna 'Online Backup':
Online Backup
0    4614
1    2429
Name: count, dtype: int64

Valores únicos en la columna 'Device Protection':
Device Protection
0    4621
1    2422
Name: count, dtype: int64

Valores únicos en la columna 'Tech Support':
Tech Support
0    4999
1    2044
Name: count, dtype: int64

Valores únicos en la columna 'Streaming TV':
Streaming TV
0    4336
1    2707
Name: count, dtype: int64

Valores únicos en la columna 'Streaming Movies':
Streaming Movies
0    4311
1    2732
Name: count, dtype: int64



In [10]:
df_limpio[columnas_servicios].isnull().sum()

Online Security      0
Online Backup        0
Device Protection    0
Tech Support         0
Streaming TV         0
Streaming Movies     0
dtype: int64

### Resultado: fusión de "No internet service"

Las 6 columnas de servicios dependientes de Internet Service fueron convertidas a binario
puro (0/1), fusionando "No" y "No internet service" en un mismo valor (0), ya que ambos
representan la ausencia del servicio. Verificado con Online Security: 5,024 en "0"
(3,498 "No" + 1,526 "No internet service" originales) y 2,019 en "1" — coincide exactamente
con los conteos de la Etapa 2, confirmando que no se perdió ningún registro en la fusión.

## Encoding de `Contract` (la decisión ordinal)

La idea es convertir la columna `Contract` a números, pero a diferencia de las anteriores, acá vamos a usar codificación ordinal en vez de one-hot — es decir, asignarle un número que respete el orden de compromiso contractual: Month-to-month < One year < Two year.

Por qué ya lo vimos clarísimo en Excel — el nivel de compromiso contractual tiene una relación directa y ordenada con el churn (42,7% → 11,3% → 2,8%, bajando escalonadamente). Cuando existe un orden de negocio real como este, la codificación ordinal le da al modelo información extra que el one-hot no captura: la distancia entre categorías. Con one-hot, "Month-to-month" y "Two year" serían simplemente "distintas", sin ninguna noción de que una implica más compromiso que la otra

In [11]:
df_limpio['Contract'] = df_limpio['Contract'].map({'Month-to-month': 0, 'One year': 1, 'Two year': 2}) 

In [12]:
df_limpio['Contract'].value_counts()

Contract
0    3875
2    1695
1    1473
Name: count, dtype: int64

In [13]:
df_limpio['Contract'].isnull().sum()

np.int64(0)

### Resultado: encoding ordinal de Contract

`Contract` fue codificada de forma ordinal (Month-to-month=0, One year=1, Two year=2),
respetando el orden creciente de compromiso contractual observado en el análisis de Excel
(42.7% → 11.3% → 2.8% de churn). Verificado: 0 valores nulos, conteos coinciden exactamente
con la exploración previa (3,875 / 1,473 / 1,695).

## One-hot encoding del resto de las categóricas
Voy convertir las columnas categóricas que no tienen orden lógico entre sus valores a one-hot encoding. Son: `Multiple Lines`,` Internet Service`, `Payment Method`.

Por qué a diferencia de `Contract`, acá no existe una jerarquía natural. Por ejemplo, no tiene sentido decir que "Fiber optic" es "más" o "menos" que "DSL" — son categorías distintas sin relación de orden entre sí. Si les asignáramos números arbitrarios (como 0, 1, 2), el modelo podría interpretar erróneamente que hay una relación de magnitud entre ellas que no existe en la realidad. El one-hot evita ese problema creando una columna binaria por cada categoría.

In [14]:
columnas_onehot = ['Multiple Lines', 'Internet Service', 'Payment Method']
df_limpio = pd.get_dummies(df_limpio, columns=columnas_onehot)

In [15]:
df_limpio.columns.tolist

<bound method IndexOpsMixin.tolist of Index(['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Tenure Months',
       'Phone Service', 'Online Security', 'Online Backup',
       'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies',
       'Contract', 'Paperless Billing', 'Monthly Charges', 'Total Charges',
       'Churn Value', 'Multiple Lines_No', 'Multiple Lines_No phone service',
       'Multiple Lines_Yes', 'Internet Service_DSL',
       'Internet Service_Fiber optic', 'Internet Service_No',
       'Payment Method_Bank transfer (automatic)',
       'Payment Method_Credit card (automatic)',
       'Payment Method_Electronic check', 'Payment Method_Mailed check'],
      dtype='str')>

In [16]:
df_limpio.shape

(7043, 27)

### Resultado: one-hot encoding

Las columnas Multiple Lines, Internet Service y Payment Method fueron convertidas a
one-hot encoding con `pd.get_dummies()`. El DataFrame pasó de 20 a 27 columnas
(10 columnas nuevas por las 3 categóricas reemplazadas, netas de las 3 originales
eliminadas). Todas las columnas quedan ahora en formato numérico/binario.

## Feature engineering — crear variables nuevas

Voy crear 3 variables nuevas a partir de las columnas existentes, tal como planeamos desde el planning original:

- Cantidad total de servicios contratados — sumar cuántos "Sí" tiene cada cliente entre los servicios (Online Security, Online Backup, Device Protection, Tech Support, Streaming TV, Streaming Movies, Phone Service)
- Cargo promedio por mes de antigüedad — `Total Charges` / `Tenure Months`
- Grupos de antigüedad — los mismos rangos que usaste en Excel (0-11, 12-23, etc.)

Por qué estas variables no están explícitas en el dataset original, pero capturan patrones que las columnas individuales no reflejan por separado. La intuición de negocio: un cliente con muchos servicios contratados está más "atado" a la empresa (más fricción para irse), y el cargo promedio mensual puede revelar inconsistencias o patrones de gasto que el cargo mensual actual solo no muestra.

### Cantidad total de servicios

In [17]:
columnas_servicios_total = ['Phone Service', 'Online Security', 'Online Backup', 'Device Protection', 'Tech Support', 'Streaming TV', 'Streaming Movies']

df_limpio['Total Services'] = df_limpio[columnas_servicios_total].sum(axis=1)

In [21]:
df_limpio['Total Services'].describe()

count    7043.000000
mean        2.941076
std         1.843899
min         0.000000
25%         1.000000
50%         3.000000
75%         4.000000
max         7.000000
Name: Total Services, dtype: float64

#### Resultado: Total_Servicios

Se creó la variable `Total_Servicios`, sumando horizontalmente (axis=1) las 7 columnas
de servicios binarios (Phone Service, Online Security, Online Backup, Device Protection,
Tech Support, Streaming TV, Streaming Movies). Rango confirmado: 0 a 7, con un promedio
de 2.94 servicios por cliente. La intuición de negocio es que a mayor cantidad de
servicios contratados, mayor "atadura" del cliente a la empresa, y por lo tanto menor
probabilidad de fuga — a confirmar con el modelo en la Etapa 4.

### Cargo promedio por mes de antigüedad

Voy a crear`Cargo_Promedio_Mensual = Total Charges / Tenure Months`

Por qué esto puede revelar inconsistencias interesantes. Por ejemplo, si el cargo promedio histórico de un cliente es mucho más bajo que su `Monthly Charges` actual, podría indicar que subió de plan recientemente (y eso podría relacionarse con el riesgo de irse).

In [22]:
import numpy as np

df_limpio['Cargo_Promedio_Mensual'] = np.where(
    df_limpio['Tenure Months'] == 0,
    0,
    df_limpio['Total Charges'] / df_limpio['Tenure Months']
)

In [23]:
df_limpio['Cargo_Promedio_Mensual'].describe()

count    7043.000000
mean       64.698218
std        30.270670
min         0.000000
25%        35.649000
50%        70.300000
75%        90.174158
max       121.400000
Name: Cargo_Promedio_Mensual, dtype: float64

#### Resultado: Cargo_Promedio_Mensual

Se creó `Cargo_Promedio_Mensual = Total Charges / Tenure Months`, manejando el caso de
división por cero con `np.where()` — los 11 clientes con `Tenure Months = 0` reciben
directamente el valor 0. Rango confirmado: 0 a 121.4, sin valores infinitos ni nulos.

### Grupos de antigüedad

Voy a crear una versión categórica de `Tenure Months`, agrupando en los mismos rangos que ya usaste en Excel (0-11, 12-23, 24-35, 36-47, 48-59, 60-72).

Por qué a veces los modelos encuentran patrones más simples y estables trabajando con rangos en vez del número exacto — por ejemplo, la diferencia real de comportamiento entre un cliente de 13 meses y uno de 14 meses probablemente sea insignificante, pero agrupados en "12-23 meses" el modelo puede captar mejor el patrón general de ese segmento. Además, esto te conecta directamente con el análisis que ya hiciste en Excel, dándole coherencia narrativa al proyecto.

In [24]:
df_limpio['Grupo_Antiguedad'] = pd.cut(
    df_limpio['Tenure Months'],
    bins=[-1, 11, 23, 35, 47, 59, 72],
    labels=['0-11', '12-23', '24-35', '36-47', '48-59', '60-72']
)

In [25]:
df_limpio['Grupo_Antiguedad'].value_counts()

Grupo_Antiguedad
0-11     2069
60-72    1483
12-23    1047
24-35     876
48-59     820
36-47     748
Name: count, dtype: int64

In [26]:
orden_antiguedad = {'0-11': 0, '12-23': 1, '24-35': 2, '36-47': 3, '48-59': 4, '60-72': 5}
df_limpio['Grupo_Antiguedad'] = df_limpio['Grupo_Antiguedad'].map(orden_antiguedad)

In [27]:
df_limpio['Grupo_Antiguedad'].value_counts()

Grupo_Antiguedad
0    2069
5    1483
1    1047
2     876
4     820
3     748
Name: count, dtype: int64

#### Resultado: Grupo_Antiguedad

Se creó `Grupo_Antiguedad`, categorizando `Tenure Months` en los mismos 6 rangos usados
en la exploración de Excel (0-11, 12-23, 24-35, 36-47, 48-59, 60-72) mediante `pd.cut()`.
El grupo más numeroso es 0-11 meses (2,069 clientes) — coincide con el segmento de mayor
riesgo de churn identificado previamente (48.3%).

Encoding de `Grupo_Antiguedad`:Se aplicó codificación ordinal a `Grupo_Antiguedad` (0 a 5, de menor a mayor antigüedad),
siguiendo el mismo criterio usado en `Contract` — el orden de negocio real (a mayor
antigüedad, menor riesgo de churn) justifica preservar la progresión numérica en vez de
usar one-hot encoding.

## Estrategia para el desbalance de clases

Ya confirmado en la Etapa 2: el dataset tiene un desbalance moderado (73.46% No Churn
vs. 26.54% Yes Churn). La estrategia definida para la Etapa 4 es:

1. **Primera opción**: usar `class_weight='balanced'` en los modelos que lo soporten
   (Regresión Logística, Random Forest) — le da más peso a los errores sobre la clase
   minoritaria (Yes) sin modificar el dataset.
2. **Alternativa a evaluar si el desempeño no es suficiente**: SMOTE, generando ejemplos
   sintéticos de la clase minoritaria.

Esta decisión no se ejecuta en este notebook — se aplica como parámetro del modelo en
la Etapa 4, ya que no requiere modificar el dataset guardado en `data/processed/`.

## Separar features (X) del target (y)

Aunque conceptualmente X e y son insumos del modelado (Etapa 4), se separan acá porque
este notebook termina guardando el dataset ya procesado en `data/processed/`. Separarlos
ahora evita repetir este paso en cada notebook que use los datos limpios — la Etapa 4
carga X e y directamente, sin necesidad de rehacer el `.drop()`.

Lo que sí es una regla fija, independientemente de en qué etapa se haga: el modelo nunca
debe entrenar viendo features y target mezclados en la misma tabla.

In [28]:
X = df_limpio.drop(columns=['Churn Value'])
y = df_limpio['Churn Value']

In [30]:
X.shape

(7043, 29)

In [33]:
y.shape

(7043,)

## Guardar el dataset procesado

In [34]:
X.to_csv('../data/processed/X_features.csv', index=False)
y.to_csv('../data/processed/y_target.csv', index=False)

In [35]:
import os
os.listdir('../data/processed/')

['X_features.csv', 'y_target.csv']

## Cierre de la Etapa 3

Se completó la preparación de datos: filtrado de columnas, corrección de tipos
(Total Charges), encoding de variables binarias y categóricas (incluyendo decisiones
justificadas de codificación ordinal para Contract y Grupo_Antiguedad), creación de
3 variables nuevas mediante feature engineering, y definición de la estrategia para el
desbalance de clases (a aplicar en la Etapa 4).

El dataset final quedó separado en `X_features.csv` (28 columnas predictoras, 7,043
registros) y `y_target.csv` (target Churn Value), guardados en `data/processed/`,
listos para ser cargados directamente en la Etapa 4 sin repetir este proceso.